# OpenRouter Reasoning Validation (This Environment)

This notebook validates reasoning-token streaming behavior for your current stack.

It covers:
- Installed package/runtime versions
- `init_chat_model` provider support (including `openrouter`)
- Available classes in `langchain_openai`
- LangChain stream chunks for OpenRouter via `ChatOpenAI`
- Raw OpenRouter SSE for `chat/completions`
- Raw OpenRouter SSE for `responses` API


In [1]:
import importlib.metadata as md

pkgs = [
    "langchain",
    "langchain-openai",
    "openai",
    "langgraph",
    "langchain-core",
    "langchain-openrouter"
]

for p in pkgs:
    try:
        print(f"{p}: {md.version(p)}")
    except Exception as e:
        print(f"{p}: ERR -> {e}")


langchain: 1.2.10
langchain-openai: 1.1.9
openai: 2.20.0
langgraph: 1.0.8
langchain-core: 1.2.13
langchain-openrouter: 0.0.2


In [2]:
from langchain.chat_models import init_chat_model
from langchain_openrouter import ChatOpenRouter


def get_llm(model_name, provider, api_key):
    if provider == "openrouter":
        # init_chat_model doesn't support this yet, so instantiate directly
        return ChatOpenRouter(model=model_name, api_key=api_key)

    # Use standard init for everything else
    return init_chat_model(model=model_name, model_provider=provider, api_key=api_key)

In [3]:
import os

In [4]:
llm = get_llm("z-ai/glm-5", "openrouter", os.environ.get("OPENROUTER_API_KEY"))

In [10]:
events = llm.astream_events(input="How many r`s are in the word `strawberry?`", version="v2")

In [11]:
# async for event in events:
#     print(event)

In [12]:
async for event in events:
    kind = event["event"]
    name = event["name"]

    # 1. Handle Node Transitions (LangGraph specific)
    if kind == "on_chain_start":
        # Check if this is a LangGraph node vs a generic chain
        if event["metadata"].get("langgraph_node"):
            print(f"\n\n[Entering Node: {name}]")

    # 2. Handle LLM Tokens (Real-time)
    elif kind == "on_chat_model_stream":
        content = event["data"]["chunk"].content
        if content:
            # Print tokens without newlines for a fluid experience
            print(f"\n[Real Token]: {content}", end="", flush=True)

        # Extract Reasoning (if using OpenRouter/DeepSeek)
        reasoning = event["data"]["chunk"].additional_kwargs.get("reasoning_content")
        if reasoning:
            print(f"\n[Thought]: {reasoning}", end="", flush=True)

    # 3. Handle Tool Execution
    elif kind == "on_tool_start":
        print(f"\n\n[Tool Start]: {name}")
        print(f"Inputs: {event['data'].get('input')}")

    elif kind == "on_tool_end":
        print(f"\n[Tool End]: {name} completed.")

    # 4. Final Response Summary
    elif kind == "on_chat_model_end":
        print("\n" + "-"*20)


[Thought]: The
[Thought]:  user
[Thought]:  is
[Thought]:  asking
[Thought]:  for
[Thought]:  the
[Thought]:  count
[Thought]:  of
[Thought]:  the
[Thought]:  letter
[Thought]:  '
[Thought]: r
[Thought]: '
[Thought]:  in
[Thought]:  the
[Thought]:  word
[Thought]:  '
[Thought]: str
[Thought]: aw
[Thought]: berry
[Thought]: '.


[Thought]: 1
[Thought]: .
[Thought]:  
[Thought]:  **
[Thought]: An
[Thought]: alyze
[Thought]:  the
[Thought]:  word
[Thought]: :**
[Thought]:  The
[Thought]:  word
[Thought]:  is
[Thought]:  "
[Thought]: str
[Thought]: aw
[Thought]: berry
[Thought]: ".

[Thought]: 2
[Thought]: .
[Thought]:  
[Thought]:  **
[Thought]: Spell
[Thought]:  it
[Thought]:  out
[Thought]: :**
[Thought]:  s
[Thought]: -t
[Thought]: -r
[Thought]: -a
[Thought]: -w
[Thought]: -b
[Thought]: -e
[Thought]: -r
[Thought]: -r
[Thought]: -y
[Thought]: .

[Thought]: 3
[Thought]: .
[Thought]:  
[Thought]:  **
[Thought]: Ident
[Thought]: ify
[Thought]:  the
[Thought]:  target
[Thought]:  letter
[T

In [2]:

providers = ["openrouter", "openai", "groq", "nvidia", "deepseek"]
for provider in providers:
    try:
        model = init_chat_model(model="z-ai/glm-5", model_provider=provider, api_key="dummy")
        print(f"{provider}: OK -> {type(model).__name__}")
    except Exception as e:
        print(f"{provider}: ERR -> {type(e).__name__}: {e}")


openrouter: ERR -> ValueError: Unsupported provider='openrouter'.

Supported model providers are: anthropic, azure_ai, azure_openai, bedrock, bedrock_converse, cohere, deepseek, fireworks, google_anthropic_vertex, google_genai, google_vertexai, groq, huggingface, ibm, mistralai, nvidia, ollama, openai, perplexity, together, upstage, xai
openai: OK -> ChatOpenAI
groq: OK -> ChatGroq
nvidia: OK -> ChatNVIDIA
deepseek: OK -> ChatDeepSeek


/home/pruthvi/projects/h-ai/mft-FastMCP-server-httpx-tools/.venv/lib/python3.11/site-packages/langchain_nvidia_ai_endpoints/_common.py:250: UserWarning: Model z-ai/glm-5 is unknown, check `available_models`. Inference may fail.
  warnings.warn(


In [ ]:
import langchain_openai as lco

interesting = [name for name in dir(lco) if "OpenRouter" in name or "ChatOpen" in name]
print(interesting)


In [ ]:
import asyncio
import os

from langchain_core.messages import HumanMessage
from langchain_openai import ChatOpenAI

OPENROUTER_API_KEY = os.environ.get("OPENROUTER_API_KEY")
if not OPENROUTER_API_KEY:
    raise RuntimeError("OPENROUTER_API_KEY is missing in environment")

async def probe_langchain_stream():
    llm = ChatOpenAI(
        model="z-ai/glm-5",
        api_key=OPENROUTER_API_KEY,
        base_url="https://openrouter.ai/api/v1",
        reasoning={"effort": "high"},
        temperature=0.0,
    )

    merged = None
    i = 0
    async for chunk in llm.astream([HumanMessage(content="How many r letters are in strawberry?")]):
        i += 1
        merged = chunk if merged is None else (merged + chunk)

        content = getattr(chunk, "content", None)
        addl = getattr(chunk, "additional_kwargs", None)
        response_meta = getattr(chunk, "response_metadata", None)

        if content or addl:
            print(f"\nchunk {i}")
            print("content=", repr(content))
            print("additional_kwargs=", repr(addl))
            print("response_metadata_keys=", list((response_meta or {}).keys()))

        if i >= 40:
            break

    print("\n=== MERGED SUMMARY ===")
    print("merged.content=", repr(getattr(merged, "content", None))[:1600])
    print("merged.additional_kwargs=", repr(getattr(merged, "additional_kwargs", None)))
    print("merged.response_metadata.keys=", list((getattr(merged, "response_metadata", {}) or {}).keys()))
    print("merged.usage_metadata=", repr(getattr(merged, "usage_metadata", None)))

asyncio.run(probe_langchain_stream())


In [ ]:
import json
import os

import httpx

OPENROUTER_API_KEY = os.environ.get("OPENROUTER_API_KEY")
if not OPENROUTER_API_KEY:
    raise RuntimeError("OPENROUTER_API_KEY is missing in environment")

url = "https://openrouter.ai/api/v1/chat/completions"
headers = {
    "Content-Type": "application/json",
    "Authorization": f"Bearer {OPENROUTER_API_KEY}",
}
payload = {
    "model": "z-ai/glm-5",
    "stream": True,
    "messages": [{"role": "user", "content": "How many r letters are in strawberry?"}],
    "reasoning": {"enabled": True},
}

shown = 0
with httpx.Client(timeout=90) as client:
    with client.stream("POST", url, headers=headers, json=payload) as resp:
        resp.raise_for_status()
        for line in resp.iter_lines():
            if not line:
                continue
            if line.startswith(":"):
                continue
            if not line.startswith("data: "):
                continue

            body = line[6:]
            if body == "[DONE]":
                print("[DONE]")
                break

            obj = json.loads(body)
            delta = (obj.get("choices") or [{}])[0].get("delta") or {}
            reasoning = delta.get("reasoning")
            content = delta.get("content")

            if reasoning is not None or content is not None:
                print({
                    "reasoning": (reasoning[:120] + "...") if isinstance(reasoning, str) and len(reasoning) > 120 else reasoning,
                    "content": (content[:120] + "...") if isinstance(content, str) and len(content) > 120 else content,
                    "has_reasoning_details": bool(delta.get("reasoning_details")),
                })
                shown += 1

            if shown >= 30:
                break


In [ ]:
import json
import os

import httpx

OPENROUTER_API_KEY = os.environ.get("OPENROUTER_API_KEY")
if not OPENROUTER_API_KEY:
    raise RuntimeError("OPENROUTER_API_KEY is missing in environment")

url = "https://openrouter.ai/api/v1/responses"
headers = {
    "Content-Type": "application/json",
    "Authorization": f"Bearer {OPENROUTER_API_KEY}",
}
payload = {
    "model": "z-ai/glm-5",
    "stream": True,
    "input": [{"role": "user", "content": "How many r letters are in strawberry?"}],
    "reasoning": {"effort": "high"},
}

shown = 0
with httpx.Client(timeout=90) as client:
    with client.stream("POST", url, headers=headers, json=payload) as resp:
        resp.raise_for_status()
        for line in resp.iter_lines():
            if not line:
                continue
            if line.startswith(":"):
                continue
            if not line.startswith("data: "):
                continue

            body = line[6:]
            if body == "[DONE]":
                print("[DONE]")
                break

            obj = json.loads(body)
            t = obj.get("type")

            if t in {"response.reasoning_text.delta", "response.output_text.delta"}:
                delta = obj.get("delta")
                if isinstance(delta, str) and len(delta) > 120:
                    delta = delta[:120] + "..."
                print({"type": t, "delta": delta})
                shown += 1

            if shown >= 40:
                break


## How To Interpret Results

- If `model_provider="openrouter"` is unsupported in your runtime, that claim is not valid for this environment.
- If LangChain chunks only show reasoning placeholders (no reasoning text), your app cannot emit true per-token `event: reasoning` from LangChain alone.
- If raw OpenRouter SSE shows reasoning deltas but LangChain does not, the gap is in adapter normalization/parsing, not OpenRouter itself.
